# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Лавренюк Артем Дмитрович

**Група:** ФІТ 4-10

**Дата:** 18.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [4]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print('Кількість унікальних користувачів:', ratings_df['user_id'].nunique())
print('Кількість унікальних фільмів:', ratings_df['item_id'].nunique())
print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', round(ratings_df['rating'].mean(), 3))
print('Кількість пропущених значень у кожному стовпці:')
display(ratings_df.isna().sum().to_frame(name='Пропущені значення'))
print('Розподіл оцінок:')
display(ratings_df['rating'].value_counts().sort_index().to_frame(name='Кількість'))


Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53
Кількість пропущених значень у кожному стовпці:


,Пропущені значення
user_id,0
item_id,0
rating,0
timestamp,0


Розподіл оцінок:


,Кількість
rating,
1.0,6110
2.0,11370
3.0,27145
4.0,34174
5.0,21201


**Короткі спостереження щодо даних:**

Набір MovieLens 100K містить 100 000 оцінок, які залишили 943 користувачі для 1 682 фільмів. Оцінки знаходяться в діапазоні від 1 до 5, а середнє значення становить приблизно 3,53. Це означає, що в наборі переважають оцінки середнього та вищого рівня, а дані можна використовувати для навчання моделей прогнозування вподобань користувачів.


## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print('Розмір навчальної вибірки:', train_df.shape)
print('Розмір тестової вибірки:', test_df.shape)


Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


In [6]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale=data.reader.rating_scale)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)
testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print('Кількість записів у train_data:', len(train_df))
print('Кількість записів у testset:', len(testset))
print('Частка навчальної вибірки:', round(len(train_df) / len(ratings_df), 3))
print('Частка тестової вибірки:', round(len(test_df) / len(ratings_df), 3))


Кількість записів у train_data: 80000
Кількість записів у testset: 20000
Частка навчальної вибірки: 0.8
Частка тестової вибірки: 0.2


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [7]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

Для SVD параметр `n_factors` визначає кількість прихованих факторів, за допомогою яких модель описує вподобання користувачів і властивості фільмів. `lr_all` задає швидкість навчання, а `reg_all` допомагає уникати перенавчання. Тому перевіряються два помірні значення для кожного параметра.

Для KNNBasic параметр `k` визначає кількість сусідів, а `min_k` — мінімальну кількість сусідів, необхідну для прогнозу. Метрики `cosine` та `pearson` дають змогу порівняти різні способи обчислення схожості, а `user_based` перемикає пошук схожості між користувачами або між фільмами. Така сітка дозволяє порівняти кілька варіантів без довільної зміни початкових даних.


## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [8]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)


In [9]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)


In [10]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('SVD')
print('Найкращий CV RMSE:', grid_search_svd.best_score['rmse'])
print('Найкращий CV MAE:', grid_search_svd.best_score['mae'])
print('Параметри за RMSE:', grid_search_svd.best_params['rmse'])

print('\nKNNBasic')
print('Найкращий CV RMSE:', grid_search_knn.best_score['rmse'])
print('Найкращий CV MAE:', grid_search_knn.best_score['mae'])
print('Параметри за RMSE:', grid_search_knn.best_params['rmse'])

cv_comparison = pd.DataFrame([
    {
        'Model': 'SVD',
        'CV RMSE': grid_search_svd.best_score['rmse'],
        'CV MAE': grid_search_svd.best_score['mae'],
        'Best parameters': str(grid_search_svd.best_params['rmse'])
    },
    {
        'Model': 'KNNBasic',
        'CV RMSE': grid_search_knn.best_score['rmse'],
        'CV MAE': grid_search_knn.best_score['mae'],
        'Best parameters': str(grid_search_knn.best_params['rmse'])
    }
]).sort_values('CV RMSE')
display(cv_comparison)
print('Краща модель за CV RMSE:', cv_comparison.iloc[0]['Model'])


SVD
Найкращий CV RMSE: 0.9431937110196347
Найкращий CV MAE: 0.7448564770305199
Параметри за RMSE: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic
Найкращий CV RMSE: 1.0295661993315566
Найкращий CV MAE: 0.8157504561887009
Параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


,Model,CV RMSE,CV MAE,Best parameters
0,SVD,0.943194,0.744856,"{'n_factors': 100, 'lr_all': 0.01, 'reg_all': ..."
1,KNNBasic,1.029566,0.815750,"{'k': 40, 'min_k': 1, 'sim_options': {'name': ..."


Краща модель за CV RMSE: SVD


**Аналіз результатів кросвалідації:**

Порівняння двох моделей виконується за найкращими значеннями RMSE та MAE, отриманими під час трьохчастинної кросвалідації на навчальній вибірці. Менші значення цих метрик означають, що прогнозовані оцінки в середньому ближчі до реальних. Модель із меншим CV RMSE обирається для формування рекомендацій; її перевагу додатково перевіряємо на незалежній тестовій вибірці.


## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [11]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse'].copy()
best_knn_params = grid_search_knn.best_params['rmse'].copy()

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print('Остаточну модель SVD навчено.')
print('Остаточну модель KNNBasic навчено.')
print('Кількість оцінок у повній навчальній вибірці:', trainset.n_ratings)


Computing the cosine similarity matrix...
Done computing similarity matrix.
Остаточну модель SVD навчено.
Остаточну модель KNNBasic навчено.
Кількість оцінок у повній навчальній вибірці: 80000


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [12]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)
knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print(f'SVD — RMSE: {svd_rmse:.4f}, MAE: {svd_mae:.4f}')
print(f'KNNBasic — RMSE: {knn_rmse:.4f}, MAE: {knn_mae:.4f}')


SVD — RMSE: 0.9194, MAE: 0.7247
KNNBasic — RMSE: 1.0120, MAE: 0.7984


In [13]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame([
    {'Model': 'SVD', 'RMSE': svd_rmse, 'MAE': svd_mae},
    {'Model': 'KNNBasic', 'RMSE': knn_rmse, 'MAE': knn_mae}
]).sort_values('RMSE').reset_index(drop=True)

display(results)
print('Модель із найменшим тестовим RMSE:', results.iloc[0]['Model'])


,Model,RMSE,MAE
0,SVD,0.919367,0.724672
1,KNNBasic,1.011953,0.798421


Модель із найменшим тестовим RMSE: SVD


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | ... | ... |
| KNNBasic | ... | ... |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [14]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
best_model = (
    best_svd
    if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse']
    else best_knn
)
best_model_name = 'SVD' if best_model is best_svd else 'KNNBasic'
print('Обрана за CV RMSE модель:', best_model_name)

try:
    user_inner_id = trainset.to_inner_uid(user_id)
except ValueError:
    user_inner_id = None
    print(f'Користувача з ID {user_id} немає у навчальній вибірці. '
          'Змініть user_id на наявного користувача.')

if user_inner_id is not None:
    rated_inner_items = {
        inner_iid
        for inner_iid, rating in trainset.ur[user_inner_id]
    }
    recommendations = []
    for inner_iid in trainset.all_items():
        if inner_iid not in rated_inner_items:
            raw_iid = trainset.to_raw_iid(inner_iid)
            predicted_rating = best_model.predict(user_id, raw_iid).est
            recommendations.append((raw_iid, predicted_rating))

    top10_recommendations = (
        pd.DataFrame(recommendations, columns=['item_id', 'predicted_rating'])
        .sort_values('predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    display(top10_recommendations)


Обрана за CV RMSE модель: SVD


,item_id,predicted_rating
0,318,4.578034
1,427,4.510453
2,603,4.455609
3,1169,4.441489
4,661,4.414566
5,479,4.390852
6,513,4.356216
7,357,4.324538
8,64,4.322364
9,313,4.300572


## 9. Перевірка виконання

Позначте після реалізації:

- [+] Дані `ml-100k` завантажено та перевірено.
- [+] Є розбиття 80/20 з фіксованим `random_state`.
- [+] `Reader`, `train_data` і `testset` створені коректно.
- [+] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [+] Для SVD і KNNBasic показано параметри та результати CV.
- [+] Моделі навчено на повній навчальній частині.
- [+] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [+] Наведено порівняльну таблицю та інтерпретацію.
- [+] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [+] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?

Для формування рекомендацій обирається модель із найменшим значенням CV RMSE за результатами трьохчастинної кросвалідації. Менше значення RMSE свідчить про меншу середню квадратичну похибку прогнозування. Порівнюються моделі SVD та KNNBasic, після чого для рекомендацій використовується модель із кращим результатом.

2. Які найкращі гіперпараметри отримано для обох моделей?

Для моделі SVD підбираються параметри n_factors, lr_all та reg_all. Для KNNBasic підбираються k, min_k і параметри метрики схожості sim_options. Найкращі значення визначаються автоматично за результатами GridSearchCV із використанням CV RMSE.

3. Які підсумкові значення RMSE та MAE на тестовій вибірці?

Підсумкові значення RMSE та MAE обчислюються окремо для моделей SVD і KNNBasic на незалежній тестовій вибірці. RMSE характеризує величину похибки прогнозування з більшим впливом значних відхилень, а MAE показує середню абсолютну різницю між прогнозованими та реальними оцінками.

4. Чи є різниця між результатами CV та тестування?

Результати кросвалідації та тестування можуть дещо відрізнятися, оскільки під час кросвалідації використовуються декілька частин навчальної вибірки, а під час фінального тестування — окрема незалежна вибірка. Невелика різниця є нормальною. Значна різниця може свідчити про перенавчання моделі або недостатню узагальнювальну здатність.

5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

Основним обмеженням рекомендаційної системи є проблема холодного старту. Для нових користувачів немає історії оцінок, на основі якої можна визначити їхні вподобання. Аналогічна проблема виникає з новими фільмами, які ще не отримали достатньої кількості оцінок. Також якість рекомендацій залежить від кількості та повноти наявних даних. Якщо користувач залишив мало оцінок, система може формувати менш точні рекомендації.

**Висновок:** Під час виконання лабораторної роботи було розглянуто принципи роботи рекомендаційних систем та досліджено два алгоритми: SVD і KNNBasic. Було завантажено набір даних MovieLens 100K, виконано його аналіз та розподіл на навчальну і тестову вибірки. За допомогою GridSearchCV було підібрано найкращі параметри для обох моделей. Після навчання алгоритмів проведено оцінювання їхньої роботи за показниками RMSE та MAE. На основі отриманих результатів було визначено ефективнішу модель та сформовано список із десяти рекомендацій для вибраного користувача. Також було розглянуто обмеження рекомендаційних систем, пов'язані з недостатньою кількістю даних про користувачів і фільми. У результаті роботи вдалося краще зрозуміти принципи побудови та використання рекомендаційних систем.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

In [15]:
# Формування висновку на основі фактично отриманих результатів\ncv_best_name = ('SVD' if grid_search_svd.best_score['rmse'] <= grid_search_knn.best_score['rmse'] else 'KNNBasic')\ntest_best_name = results.iloc[0]['Model']\nsvd_cv_rmse = grid_search_svd.best_score['rmse']\nknn_cv_rmse = grid_search_knn.best_score['rmse']\nprint('Висновок: У лабораторній роботі було досліджено рекомендаційні моделі SVD та KNNBasic на наборі MovieLens 100K. '      f'За результатами кросвалідації кращою стала модель {cv_best_name}, оскільки вона має менше значення CV RMSE. '      f'Для SVD найкращі параметри: {grid_search_svd.best_params["rmse"]}. '      f'Для KNNBasic найкращі параметри: {grid_search_knn.best_params["rmse"]}. '      f'На тестовій вибірці SVD отримала RMSE={svd_rmse:.4f} та MAE={svd_mae:.4f}, '      f'а KNNBasic — RMSE={knn_rmse:.4f} та MAE={knn_mae:.4f}. '      f'Найменше тестове RMSE має модель {test_best_name}. '      'Порівняння CV і тестових показників дає змогу оцінити, наскільки стабільно модель працює на нових оцінках. '      'Сформований список містить фільми, які користувач ще не оцінював у навчальній вибірці. '      'Обмеженням системи є складність рекомендацій для нових користувачів або фільмів, для яких ще немає історії оцінок.')

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [16]:

# Додаткове завдання
# Власний рекомендаційний алгоритм на основі схожості користувачів

import numpy as np
import pandas as pd

# Створення матриці оцінок користувачів
user_item_matrix = train_df.pivot_table(
    index='user_id',
    columns='item_id',
    values='rating'
)

# Перевірка наявності користувача
if user_id not in user_item_matrix.index:
    print(f'Користувача {user_id} немає у навчальній вибірці.')
else:
    # Оцінки вибраного користувача
    target_ratings = user_item_matrix.loc[user_id]

    # Користувачі, які мають спільні оцінені фільми
    recommendations_custom = []

    for item_id in user_item_matrix.columns:

        # Пропускаємо вже оцінені фільми
        if pd.notna(target_ratings[item_id]):
            continue

        # Користувачі, які оцінили поточний фільм
        other_users = user_item_matrix[
            user_item_matrix[item_id].notna()
        ]

        similarities = []

        for other_id, other_ratings in other_users.iterrows():

            if other_id == user_id:
                continue

            # Спільні фільми двох користувачів
            common = target_ratings.notna() & other_ratings.notna()

            if common.sum() == 0:
                continue

            # Косинусна подібність
            a = target_ratings[common].to_numpy(dtype=float)
            b = other_ratings[common].to_numpy(dtype=float)

            denominator = np.linalg.norm(a) * np.linalg.norm(b)

            if denominator == 0:
                continue

            similarity = np.dot(a, b) / denominator

            similarities.append(
                (similarity, other_ratings[item_id])
            )

        # Прогнозування оцінки
        if similarities:
            weights = np.array([x[0] for x in similarities])
            ratings = np.array([x[1] for x in similarities])

            if np.abs(weights).sum() > 0:
                predicted = np.dot(weights, ratings) / np.abs(weights).sum()
            else:
                predicted = ratings.mean()
        else:
            # Якщо немає схожих користувачів
            predicted = train_df[
                train_df['item_id'] == item_id
            ]['rating'].mean()

        recommendations_custom.append(
            (item_id, predicted)
        )

    # Формування топ-10 рекомендацій
    custom_recommendations = pd.DataFrame(
        recommendations_custom,
        columns=['item_id', 'predicted_rating']
    )

    custom_recommendations = (
        custom_recommendations
        .sort_values('predicted_rating', ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    print('Топ-10 рекомендацій власного алгоритму:')
    display(custom_recommendations)


Топ-10 рекомендацій власного алгоритму:


,item_id,predicted_rating
0,1642,5.0
1,850,5.0
2,1653,5.0
3,1612,5.0
4,1467,5.0
5,1629,5.0
6,1500,5.0
7,1201,5.0
8,1306,5.0
9,1293,5.0


**Пояснення до додаткового завдання:**

В додатковому завданні було реалізовано власний рекомендаційний алгоритм, який визначає схожість між користувачами за їхніми оцінками фільмів. Для цього використовується косинусна подібність. На основі оцінок схожих користувачів система прогнозує, які фільми можуть зацікавити вибраного користувача. Після обчислень формується список із десяти фільмів із найвищими прогнозованими оцінками. Такий підхід дозволяє створити просту рекомендаційну систему без використання готових алгоритмів бібліотеки Surprise.